# 01 数据入库

把行情和分红相关数据存进本地 SQLite 数据库 `basis.db`，后面两个 notebook 只读这个库：

```
data/（手工导出的 Excel/CSV）─┐
                              ├→ basis.db ─→ 02_分红预测 ─→ 03_基差分析
聚宽（分红/权重/价格/财务）───┘
```

所有写入都是 `INSERT OR REPLACE`，重复运行不会产生脏数据。

In [ ]:
import os, re, sys, time, sqlite3
from bisect import bisect_left
from datetime import datetime, timedelta
from typing import Optional

import numpy as np
import pandas as pd

ROOT = os.getcwd()
DB_PATH = os.path.join(ROOT, "basis.db")
DATA_DIR = os.path.join(ROOT, "data")

PRODUCTS = {
    "50":   {"index": "000016.SH", "fut": "IH", "divpt": "h50060.CSI", "etf": "510050.SH", "idx_opt": "HO", "label": "上证50"},
    "300":  {"index": "000300.SH", "fut": "IF", "divpt": "h30401.CSI", "etf": "510300.SH", "idx_opt": "IO", "label": "沪深300"},
    "500":  {"index": "000905.SH", "fut": "IC", "divpt": "h30402.CSI", "etf": "510500.SH", "idx_opt": None, "label": "中证500"},
    "1000": {"index": "000852.SH", "fut": "IM", "divpt": None,         "etf": None,        "idx_opt": "MO", "label": "中证1000"},
}

SLOTS = ["00", "01", "02", "03"]


SLOT_LABEL = {"00": "当月", "01": "次月", "02": "季月1", "03": "季月2"}

RUN_IMPORT = True     # 从 data/ 导入行情（幂等，可重复跑）
RUN_JQ = False        # 聚宽取数，需要 .env 或环境变量里的 JQ_ACCOUNT / JQ_PASS
JQ_FULL = False       # True = 从 2010 年全量重取（约占 2 亿额度的 6%）

## 数据库结构

每张表的主键都含 `(product, date, …)`，`upsert` 按主键整行覆盖。

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS index_daily (
    product TEXT, date TEXT, close REAL, PRIMARY KEY (product, date));

-- contract 为连续合约位次(IF00..IF03)，contract_code 为当期实际合约(IF2409)
CREATE TABLE IF NOT EXISTS futures_daily (
    product TEXT, date TEXT, contract TEXT, contract_code TEXT,
    close REAL, settle REAL, open_interest REAL, volume REAL, turnover REAL,
    last_trading_date TEXT, days INTEGER, div_points REAL,
    PRIMARY KEY (product, date, contract));

-- 中证股息点指数，区间差值即已实现分红点数
CREATE TABLE IF NOT EXISTS dividend_point_index (
    product TEXT, date TEXT, value REAL, PRIMARY KEY (product, date));

CREATE TABLE IF NOT EXISTS etf_daily (
    product TEXT, date TEXT, close REAL, PRIMARY KEY (product, date));

CREATE TABLE IF NOT EXISTS option_quotes (
    product TEXT, date TEXT, option_code TEXT, contract_name TEXT,
    close REAL, volume REAL, notional REAL,
    PRIMARY KEY (product, date, option_code));

-- 股指期权(IO/MO/HO)逐合约行情。标的直接是指数，PCP反推不需要ETF换算、
-- 不需要ETF自身分红调整
CREATE TABLE IF NOT EXISTS index_option_quotes (
    product TEXT, date TEXT, option_code TEXT,
    contract_month TEXT, opt_type TEXT, strike REAL,
    close REAL, volume REAL, turnover REAL,
    PRIMARY KEY (product, date, option_code));

CREATE TABLE IF NOT EXISTS shibor (
    date TEXT, tenor TEXT, value REAL, PRIMARY KEY (date, tenor));

CREATE TABLE IF NOT EXISTS trading_calendar (date TEXT PRIMARY KEY);

-- 成分股分红事件（聚宽 STK_XR_XD），同一 (品种,股票,除息日) 唯一。
-- 四个公告日逐事件、point-in-time：
--   plan_date 预案 / sgm_date 股东大会 / impl_date 实施公告 / reg_date 股权登记
-- 后三个可能为空 —— 空本身是有效信息，表示该事件当时还没走到那一步，
-- 德邦报告的除息日预测正是按「走到哪一步」分档的，不要 dropna
CREATE TABLE IF NOT EXISTS dividend_announcement (
    product     TEXT, stock_code TEXT, ex_date TEXT,
    div_per_share REAL, div_total REAL, shares REAL,
    weight REAL, price REAL, stage TEXT,
    plan_date TEXT, sgm_date TEXT, impl_date TEXT, reg_date TEXT,
    report_date TEXT, snapshot_date TEXT,
    PRIMARY KEY (product, stock_code, ex_date));

-- 已出预案、尚未定除息日的分红（聚宽快照）。主表主键含 ex_date 放不进去，单独存。
-- 每次 jq-backfill 按品种整体替换；snapshot_date 为取数日，状态以该日为准
CREATE TABLE IF NOT EXISTS dividend_pending (
    product TEXT, stock_code TEXT, report_date TEXT, plan_date TEXT,
    div_per_share REAL, stage TEXT, sgm_date TEXT, impl_date TEXT, cancel_date TEXT,
    snapshot_date TEXT,
    PRIMARY KEY (product, stock_code, report_date, plan_date));

-- 归母净利润，用于德邦报告「路B」：分红总额 = 净利润 × 股利支付率（尚未实现）。
-- point-in-time 过滤必须用 pub_date 而不是 report_date
CREATE TABLE IF NOT EXISTS stock_financials (
    stock_code TEXT, report_date TEXT, pub_date TEXT,
    np_parent REAL, net_profit REAL,
    PRIMARY KEY (stock_code, report_date));

-- 成分股不复权收盘价，只存「月末 + 除息日」网格（weight 列已弃用，权重看 index_weight）
CREATE TABLE IF NOT EXISTS constituent_daily (
    product TEXT, date TEXT, stock_code TEXT,
    close REAL, weight REAL,
    PRIMARY KEY (product, date, stock_code));

-- 指数成分股月末权重（聚宽 get_index_weights）
CREATE TABLE IF NOT EXISTS index_weight (
    product TEXT, date TEXT, stock_code TEXT, weight REAL,
    PRIMARY KEY (product, date, stock_code));

-- 每个交易日、每个合约位次，预测从当日到该合约到期日之间的累计分红点数。
--   futures_daily.div_points = 事后真值，含前视偏差，不能回测
--   pred_points              = 当日可知信息下的预测值，可用于回测
CREATE TABLE IF NOT EXISTS dividend_forecast_daily (
    product TEXT, date TEXT, slot TEXT,
    pred_points REAL, n_stock INTEGER,
    PRIMARY KEY (product, date, slot));

-- 主键 (product, date, ...) 本身就是索引，按品种/日期查询不需要再建。
-- 只有 stock_financials 按 pub_date 过滤时用得上额外索引
CREATE INDEX IF NOT EXISTS ix_fin ON stock_financials(stock_code, pub_date);
"""


DATE_COL = {"dividend_announcement": "ex_date", "dividend_pending": "plan_date", "stock_financials": "pub_date"}


def connect(path: str = DB_PATH) -> sqlite3.Connection:
    c = sqlite3.connect(path)
    c.execute("PRAGMA journal_mode=WAL")
    c.execute("PRAGMA synchronous=NORMAL")
    c.executescript(SCHEMA)
    _migrate(c)
    return c


def _migrate(conn):
    """给老库补上后加的列。幂等，不动已有数据"""
    have = {r[1] for r in conn.execute("PRAGMA table_info(dividend_announcement)")}
    for col in ["sgm_date", "impl_date", "reg_date", "report_date"]:
        if col not in have:
            conn.execute(f"ALTER TABLE dividend_announcement ADD COLUMN {col} TEXT")
    if "cancel_date" not in {r[1] for r in conn.execute("PRAGMA table_info(dividend_pending)")}:
        conn.execute("ALTER TABLE dividend_pending ADD COLUMN cancel_date TEXT")
    conn.commit()


_OBSOLETE = ["DROP INDEX IF EXISTS ix_fut", "DROP INDEX IF EXISTS ix_con", "DROP INDEX IF EXISTS ix_con_code",
             "DROP INDEX IF EXISTS ix_iw", "DROP INDEX IF EXISTS ix_opt", "DROP INDEX IF EXISTS ix_iopt",
             "DROP INDEX IF EXISTS ix_fcst", "DROP TABLE IF EXISTS etf_distribution",
             "DROP TABLE IF EXISTS update_log"]


def vacuum(path: str = DB_PATH):
    """删冗余索引/弃用表，再 VACUUM 回收空间。需要约等于库大小的临时磁盘空间"""
    import os
    before = os.path.getsize(path)
    conn = sqlite3.connect(path)
    for sql in _OBSOLETE:
        conn.execute(sql)
    conn.commit()
    conn.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    conn.execute("VACUUM")
    conn.close()
    after = os.path.getsize(path)
    print(f"  {before/2**30:.2f} GB → {after/2**30:.2f} GB")


def to_date_str(x):
    """统一为 YYYY-MM-DD 字符串（标量或 Series）"""
    t = pd.to_datetime(x, errors="coerce")
    if isinstance(t, pd.Series):
        return t.dt.strftime("%Y-%m-%d")
    return None if pd.isna(t) else t.strftime("%Y-%m-%d")


def upsert(conn, table: str, df: pd.DataFrame) -> int:
    """幂等写入。主键冲突整行覆盖，重跑安全。

    注意是【整行】覆盖：df 里没有的列会被写成 NULL。
    """
    if df is None or df.empty:
        return 0
    df = df.dropna(how="all")
    if df.empty:
        return 0
    cols = list(df.columns)
    sql = f"INSERT OR REPLACE INTO {table} ({','.join(cols)}) VALUES ({','.join('?' * len(cols))})"
    rows = [tuple(None if pd.isna(v) else v for v in r) for r in df.itertuples(index=False)]
    conn.executemany(sql, rows)
    conn.commit()
    return len(rows)


def last_date(conn, table: str, product: str) -> Optional[str]:
    """某表某品种的最新日期，增量更新的起点依据"""
    dc = DATE_COL.get(table, "date")
    try:
        r = conn.execute(f"SELECT MAX({dc}) FROM {table} WHERE product=?", (product,)).fetchone()
        return r[0] if r and r[0] else None
    except sqlite3.OperationalError:
        return None


def next_day(d: Optional[str], default="2010-01-01") -> str:
    return default if not d else (pd.to_datetime(d) + timedelta(days=1)).strftime("%Y-%m-%d")


def status(conn) -> pd.DataFrame:
    rows = []
    for t in ["index_daily", "futures_daily", "dividend_point_index", "etf_daily",
              "option_quotes", "index_option_quotes", "dividend_announcement", "dividend_pending",
              "constituent_daily", "index_weight", "dividend_forecast_daily"]:
        dc = DATE_COL.get(t, "date")
        for _, r in pd.read_sql(
                f"SELECT product, COUNT(*) n, MIN({dc}) a, MAX({dc}) b FROM {t} GROUP BY product",
                conn).iterrows():
            rows.append({"表": t, "品种": r["product"], "行数": r["n"], "起": r["a"], "止": r["b"]})
    for t in ["stock_financials", "shibor", "trading_calendar"]:
        dc = DATE_COL.get(t, "date")
        r = conn.execute(f"SELECT COUNT(*), MIN({dc}), MAX({dc}) FROM {t}").fetchone()
        rows.append({"表": t, "品种": "-", "行数": r[0], "起": r[1], "止": r[2]})
    return pd.DataFrame(rows)

In [ ]:
conn = connect(DB_PATH)
status(conn)

## 一、从 data/ 导入行情

文件约定：`{IF,IH,IC,IM}_combine.csv`（指数/期货/股息点/到期日/分红点数）、`{品种}.xlsx`（ETF 行情、SHIBOR、ETF 期权），
股指期权放在独立文件 `{品种}期权.xlsx` 的「期权数据」sheet，或主文件的「期权数据_股指」sheet。

In [ ]:
_d = to_date_str


def import_combine(conn, path: str, product: str, verbose=True) -> dict:
    """导入 {IF,IH,IC,IM}_combine.csv（双层表头）

    提供：指数收盘、股息点指数、四个合约位次的行情/最后交易日/区间分红点数。
    """
    raw = pd.read_csv(path, header=[0, 1])
    raw.columns = [f"{str(a).strip()}|{str(b).strip()}" for a, b in raw.columns]
    raw = raw.rename(columns={raw.columns[0]: "date"})
    raw["date"] = _d(raw["date"])
    raw = raw.dropna(subset=["date"])
    cfg, stat = PRODUCTS[product], {}
    num = lambda f, t: pd.to_numeric(raw[f"{f}|{t}"], errors="coerce") if f"{f}|{t}" in raw.columns else None

    s = num("s_dq_close", cfg["index"])
    if s is not None:
        stat["index"] = upsert(conn, "index_daily", pd.DataFrame(
            {"product": product, "date": raw["date"], "close": s}).dropna(subset=["close"]))

    if cfg["divpt"]:
        v = num("s_dq_close", cfg["divpt"])
        if v is not None:
            stat["divpt"] = upsert(conn, "dividend_point_index", pd.DataFrame(
                {"product": product, "date": raw["date"], "value": v}).dropna(subset=["value"]))

    upsert(conn, "trading_calendar", pd.DataFrame({"date": raw["date"].unique()}))

    frames = []
    for i in SLOTS:
        tag = f"{cfg['fut']}{i}"
        if f"CLOSEPRICE|{tag}" not in raw.columns:
            continue
        frames.append(pd.DataFrame({
            "product": product, "date": raw["date"], "contract": tag,
            "contract_code": raw.get(f"CONTRACTCODE|{tag}"),
            "close": num("CLOSEPRICE", tag), "settle": num("SETTLEPRICE", tag),
            "open_interest": num("OPENINTEREST", tag), "volume": num("TURNOVERVOLUME", tag),
            "turnover": num("TURNOVERVALUE", tag),
            "last_trading_date": _d(raw.get(f"LASTTRADINGDATE|{tag}")),
            "days": num("days", tag), "div_points": num("Div", tag),
        }).dropna(subset=["close"]))
    if frames:
        stat["futures"] = upsert(conn, "futures_daily", pd.concat(frames, ignore_index=True))

    if verbose:
        print(f"  [{product}] {os.path.basename(path)}: " + ", ".join(f"{k}={v}" for k, v in stat.items()))
    return stat


def import_excel(conn, path: str, product: str, verbose=True) -> dict:
    """导入 {product}.xlsx 中 combine 没有的部分：ETF行情 / SHIBOR / ETF期权行情"""
    xls = pd.ExcelFile(path)
    sheet = lambda n: pd.read_excel(xls, sheet_name=n) if n in xls.sheet_names else None
    stat = {}

    df = sheet(f"{product}ETF行情")
    if df is not None and {"date", "etf_close"} <= set(df.columns):
        stat["etf"] = upsert(conn, "etf_daily", pd.DataFrame({
            "product": product, "date": _d(df["date"]),
            "close": pd.to_numeric(df["etf_close"], errors="coerce")}).dropna())

    df = sheet(f"{product}SHIBOR")
    if df is not None and "date" in df.columns:
        rs = [pd.DataFrame({"date": _d(df["date"]), "tenor": str(c),
                            "value": pd.to_numeric(df[c], errors="coerce")})
              for c in df.columns if str(c).lower().startswith("shibor")]
        if rs:
            stat["shibor"] = upsert(conn, "shibor", pd.concat(rs, ignore_index=True).dropna())

    df = sheet(f"{product}交易日历")
    if df is not None and "date" in df.columns:
        upsert(conn, "trading_calendar", pd.DataFrame({"date": _d(df["date"])}).dropna())

    df = sheet("期权数据")
    if df is None:
        df = sheet(f"{product}期权行情")
    if df is not None:
        df = df.rename(columns={"日期": "date", "合约代码": "option_code", "收盘价": "close",
                                "成交量": "volume", "成交额": "notional", "合约名称": "contract_name"})
        if {"date", "option_code", "close"} <= set(df.columns):
            q = pd.DataFrame({
                "product": product, "date": _d(df["date"]),
                "option_code": df["option_code"].astype(str).str.strip(),
                "contract_name": df.get("contract_name"),
                "close": pd.to_numeric(df["close"], errors="coerce"),
                "volume": pd.to_numeric(df.get("volume"), errors="coerce"),
                "notional": pd.to_numeric(df.get("notional"), errors="coerce"),
            }).dropna(subset=["date", "option_code", "close"])
            q = q.drop_duplicates(subset=["product", "date", "option_code"], keep="last")
            stat["option"] = upsert(conn, "option_quotes", q)

    if verbose:
        print(f"  [{product}] {os.path.basename(path)}: " +
              (", ".join(f"{k}={v}" for k, v in stat.items()) or "无可导入内容"))
    return stat


IDX_OPT_CODE_RE = re.compile(r"^[A-Z]{1,2}(\d{4})-?([CP])-?(\d+(?:\.\d+)?)(?:\.CFE)?$")


def parse_index_option_codes(codes: pd.Series) -> pd.DataFrame:
    """合约代码 → contract_month / opt_type / strike 三列（解析失败为空）"""
    p = codes.astype(str).str.strip().str.extract(IDX_OPT_CODE_RE)
    return pd.DataFrame({"contract_month": p[0], "opt_type": p[1],
                         "strike": pd.to_numeric(p[2], errors="coerce")}, index=codes.index)


def import_index_options(conn, path: str, product: str, sheet_name=None, verbose=True) -> dict:
    """导入股指期权(IO/HO/MO)逐合约行情，长表格式：
    合约代码/合约名称/日期/收盘价/成交量/成交额/(状态)

    sheet_name 不传时依次尝试 "期权数据_股指"（挂在 {product}.xlsx 里）
    和 "期权数据"（独立文件 {product}期权.xlsx）。
    """
    xls = pd.ExcelFile(path)
    candidates = [sheet_name] if sheet_name else ["期权数据_股指", "期权数据"]
    name = next((n for n in candidates if n in xls.sheet_names), None)
    if name is None:
        if verbose:
            print(f"  [{product}] {os.path.basename(path)}: 没找到股指期权sheet（试过 {candidates}），跳过")
        return {}
    df = pd.read_excel(xls, sheet_name=name)

    df = df.rename(columns={"日期": "date", "合约代码": "option_code", "收盘价": "close",
                            "成交量": "volume", "成交额": "turnover", "合约名称": "contract_name"})
    need = {"date", "option_code", "close"}
    if not need <= set(df.columns):
        if verbose:
            print(f"  [{product}] {os.path.basename(path)}: 股指期权sheet缺列 {need - set(df.columns)}")
        return {}

    codes = df["option_code"].astype(str).str.strip()
    parsed = parse_index_option_codes(codes)
    bad = parsed["contract_month"].isna().sum()
    if bad and verbose:
        sample = codes[parsed["contract_month"].isna()].head(3).tolist()
        print(f"  [{product}] 警告：{bad} 行合约代码解析失败（示例 {sample}），"
              f"这些行 contract_month/opt_type/strike 会是空值")

    q = pd.concat([pd.DataFrame({"product": product, "date": _d(df["date"]), "option_code": codes}),
                   parsed,
                   pd.DataFrame({"close": pd.to_numeric(df["close"], errors="coerce"),
                                 "volume": pd.to_numeric(df.get("volume"), errors="coerce"),
                                 "turnover": pd.to_numeric(df.get("turnover"), errors="coerce")})],
                  axis=1).dropna(subset=["date", "option_code", "close"])
    q = q.drop_duplicates(subset=["product", "date", "option_code"], keep="last")
    n = upsert(conn, "index_option_quotes", q)
    if verbose:
        print(f"  [{product}] {os.path.basename(path)}: index_option={n}")
    return {"index_option": n}


def init_all(conn, data_dir: str):
    """把 data/ 下能认出来的文件全部导入"""
    print(f"导入 combine CSV（指数/期货/股息点/到期日/分红点数）")
    for p, cfg in PRODUCTS.items():
        for c in [f"{cfg['fut']}_combine.csv", f"{p}_combine.csv"]:
            fp = os.path.join(data_dir, c)
            if os.path.exists(fp):
                import_combine(conn, fp, p)
                break
        else:
            print(f"  [{p}] 未找到 {cfg['fut']}_combine.csv")

    print("\n导入 Excel（ETF行情/SHIBOR/期权行情）")
    for p in PRODUCTS:
        fp = os.path.join(data_dir, f"{p}.xlsx")
        if os.path.exists(fp):
            import_excel(conn, fp, p)
            # 股指期权可能挂在 {p}.xlsx 的"期权数据_股指"sheet（50的做法）；大多数品种没有，不提示
            import_index_options(conn, fp, p, sheet_name="期权数据_股指", verbose=False)
        else:
            print(f"  [{p}] 未找到 {p}.xlsx")
        # 也可能是独立文件 {p}期权.xlsx（300/1000的做法）
        fp_opt = os.path.join(data_dir, f"{p}期权.xlsx")
        if os.path.exists(fp_opt):
            import_index_options(conn, fp_opt, p, sheet_name="期权数据")

In [ ]:
if RUN_IMPORT:
    init_all(conn, DATA_DIR)
status(conn)

## 二、聚宽取数

默认增量：从库内最新除息日往回退 400 天重取，已入库但当时还没发实施公告的事件也会被更新。
凭据只从环境变量（或项目根目录的 `.env`）读，不会写进任何文件。

In [ ]:
def load_env(path=os.path.join(ROOT, ".env")):
    """读取 .env 里的 KEY=VALUE 到环境变量（已存在的不覆盖）"""
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            k, _, v = line.strip().partition("=")
            if k and not k.startswith("#") and v:
                os.environ.setdefault(k.strip(), v.strip().strip("'\""))


def parse_products(items):
    """None/空 = 全部；也认 "300,50" 这种写法"""
    if not items:
        return list(PRODUCTS)
    if isinstance(items, str):
        items = [items]
    prods = [x.strip() for item in items for x in str(item).split(",") if x.strip()]
    bad = [p for p in prods if p not in PRODUCTS]
    assert not bad, f"未知品种 {bad}，可选: {list(PRODUCTS)}"
    return prods


jq = finance = query = None
INCREMENTAL_LOOKBACK_DAYS = 400


def _import_jq():
    global jq, finance, query
    if jq is None:
        try:
            import jqdatasdk
            from jqdatasdk import finance as _f, query as _q
        except ImportError:
            raise RuntimeError("需要 jqdatasdk：pip install jqdatasdk")
        jq, finance, query = jqdatasdk, _f, _q


def to_jq(code: str) -> str:
    return code.replace(".SH", ".XSHG").replace(".SZ", ".XSHE")


def from_jq(code: str) -> str:
    return code.replace(".XSHG", ".SH").replace(".XSHE", ".SZ")


def auth():
    """登录聚宽。凭据只从环境变量读"""
    _import_jq()
    acc = os.environ.get("JQ_ACCOUNT")
    pwd = os.environ.get("JQ_PASS")
    if not acc or not pwd:
        raise RuntimeError("缺少凭据：请先设置环境变量 JQ_ACCOUNT 和 JQ_PASS\n"
                 "  export JQ_ACCOUNT='你的账号'; export JQ_PASS='你的密码'")
    jq.auth(acc, pwd)
    try:
        q = jq.get_query_count()
        print(f"  ✓ 登录成功，剩余查询额度: {q}")
    except Exception:
        print("  ✓ 登录成功")

In [ ]:
EXDATE_CANDIDATES = ["a_xr_date", "xr_date", "ex_date"]


CASHDIV_CANDIDATES = ["bonus_ratio_rmb", "per_cash_div", "dividend_amount",
                      "cash_dividend", "per_dividend", "bonus_cash"]


STAGE_CANDIDATES = ["plan_progress", "progress"]


PLAN_DATE_CANDIDATES = ["board_plan_pub_date", "board_plan_dividend_pubdate",
                        "plan_pub_date"]


SGM_DATE_CANDIDATES = ["shareholders_plan_pub_date"]


IMPL_DATE_CANDIDATES = ["implementation_pub_date"]


REG_DATE_CANDIDATES = ["a_registration_date"]


REPORT_DATE_CANDIDATES = ["report_date"]


CANCEL_DATE_CANDIDATES = ["bonus_cancel_pub_date"]


SHARE_BASE_CANDIDATES = ["distributed_share_base_implement",
                         "distributed_share_base_shareholders",
                         "distributed_share_base_board"]


DIV_TOTAL_CANDIDATES = ["bonus_amount_rmb"]


def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def fetch_dividends(codes_jq, start, end, pending_since=None):
    """STK_XR_XD 分红送股事件。按批查询 → (有除息日的事件, 尚无除息日的在途预案)

    pending_since：在途预案只保留预案日不早于它的，太老的多半是早已停止实施、源里没更新状态。

    只保留现金分红(div_per_share>0)，送转股对指数点位的影响不在这里处理。

    注意 run_query 单次最多返回5000行：300只/批时全历史事件约1.2万行，
    会被静默截断（茅台等排在后面的股票整段丢失，已实测复现）。
    因此批缩到50只/批，且每批内用 offset 分页循环取到取空为止。
    """
    frames, B, PAGE = [], 50, 5000
    for i in range(0, len(codes_jq), B):
        chunk = codes_jq[i:i + B]
        off, rows = 0, 0
        while True:
            q = (query(finance.STK_XR_XD)
                 .filter(finance.STK_XR_XD.code.in_(chunk))
                 .order_by(finance.STK_XR_XD.id)
                 .limit(PAGE).offset(off))
            try:
                df = finance.run_query(q)
            except Exception:
                # 个别版本 run_query 不在 finance 下，退回顶层
                df = jq.get_fundamentals(q)
            if df is None or df.empty:
                break
            frames.append(df)
            rows += len(df)
            if len(df) < PAGE:
                break
            off += PAGE
            print(f"    ℹ 批次{i//B} 超{PAGE}行，继续翻页(offset={off})")
            time.sleep(0.2)
        if rows >= PAGE:
            print(f"    ⚠ 批次{i//B} 取满{rows}行，如仍有截断请缩小批次")
        time.sleep(0.2)   # 控制频率，别把额度瞬间打满
    if not frames:
        return pd.DataFrame(), pd.DataFrame()
    frames = [f for f in frames if len(f) and not f.isna().all().all()]
    if not frames:
        return pd.DataFrame(), pd.DataFrame()
    d = pd.concat(frames, ignore_index=True)

    c_code = "code" if "code" in d.columns else _pick_col(d, ["stock_code", "thscode"])
    c_ex = _pick_col(d, EXDATE_CANDIDATES)
    c_div = _pick_col(d, CASHDIV_CANDIDATES)
    if c_code is None or c_ex is None:
        print(f"    ⚠ STK_XR_XD 缺关键列(code/除息日)，现有列: {list(d.columns)}")
        return pd.DataFrame(), pd.DataFrame()

    out = pd.DataFrame({
        "stock_code": d[c_code].astype(str).map(from_jq),
        "ex_date": pd.to_datetime(d[c_ex], errors="coerce").dt.strftime("%Y-%m-%d"),
    })
    # 每股分红。多数版本口径是"每10股派X元"，probe 确认后再定折算系数
    if c_div is not None:
        out["div_per_share"] = pd.to_numeric(d[c_div], errors="coerce") / 10.0
    else:
        print(f"    ⚠ 没找到现金分红列，候选{CASHDIV_CANDIDATES}都不在: {list(d.columns)}")
        out["div_per_share"] = np.nan
    c_stage = _pick_col(d, STAGE_CANDIDATES)
    out["stage"] = d[c_stage].astype(str).replace({"nan": None, "None": None}) if c_stage else None

    # 四个公告日：预案→股东大会→实施→登记。空值不是缺数据，是「当时还没走到这步」，
    # 德邦报告的除息日预测正是按走到哪一步分档，所以原样保留、不要 dropna
    for col, cands in [("plan_date", PLAN_DATE_CANDIDATES),
                       ("sgm_date", SGM_DATE_CANDIDATES),
                       ("impl_date", IMPL_DATE_CANDIDATES),
                       ("reg_date", REG_DATE_CANDIDATES),
                       ("report_date", REPORT_DATE_CANDIDATES),
                       ("cancel_date", CANCEL_DATE_CANDIDATES)]:
        c = _pick_col(d, cands)
        out[col] = (pd.to_datetime(d[c], errors="coerce").dt.strftime("%Y-%m-%d")
                    if c else None)

    # 分配股本基数：源单位是【万股】，×1e4 换成股，好和每股分红直接相乘
    c_sb = _pick_col(d, SHARE_BASE_CANDIDATES)
    out["shares"] = pd.to_numeric(d[c_sb], errors="coerce") * 1e4 if c_sb else np.nan
    # 分红总额：优先用源字段；缺失时用 每股分红×股本 兜底
    # bonus_amount_rmb 源单位是【万元】（已用茅台验证：3,503,257万 = 350.3亿，
    # 与 每股28.02×12.5亿股 吻合），×1e4 换成元
    c_dt = _pick_col(d, DIV_TOTAL_CANDIDATES)
    out["div_total"] = pd.to_numeric(d[c_dt], errors="coerce") * 1e4 if c_dt else np.nan
    out["div_total"] = out["div_total"].fillna(out["div_per_share"] * out["shares"])

    # 已出预案、还没定除息日：单独返回，存 dividend_pending（主表主键含 ex_date，放不进去）
    pend = out[out["ex_date"].isna() & out["plan_date"].notna() & out["report_date"].notna()]
    if pending_since:
        pend = pend[pend["plan_date"] >= pending_since]
    if len(pend):
        keys = ["stock_code", "report_date", "plan_date"]
        pend = (pend.groupby(keys, as_index=False)["div_per_share"].sum(min_count=1)
                .merge(pend.groupby(keys, as_index=False)[["stage", "sgm_date", "impl_date", "cancel_date"]].first(), on=keys))

    out = out.dropna(subset=["ex_date"])
    out = out[(out["ex_date"] >= start) & (out["ex_date"] <= end)]
    # 同一只股票、同一除息日可能有【多笔】分红（普通股息 + 特别股息，
    # 或年报与特别分红撞在一天）。表主键是 (product, stock_code, ex_date)，
    # 直接 drop_duplicates 会丢掉其中一笔 —— 中国平安2018-06-07 就是这样：
    # 末期股息1.00 + 30周年特别股息0.20，只留下0.20，上证50当天少算6.25点。
    # 正确做法是按金额【求和】，日期类字段取最早（最先公告的那笔）。
    num = out.groupby(["stock_code", "ex_date"], as_index=False)[
        ["div_per_share", "div_total"]].sum(min_count=1)
    other = [c for c in out.columns if c not in
             ("stock_code", "ex_date", "div_per_share", "div_total")]
    first = out.sort_values("plan_date").groupby(
        ["stock_code", "ex_date"], as_index=False)[other].first()
    merged = num.merge(first, on=["stock_code", "ex_date"], how="left")
    n_dup = len(out) - len(merged)
    if n_dup:
        print(f"    合并同除息日多笔分红 {n_dup} 组（普通+特别股息等）")
    return merged, pend

In [ ]:
def sample_dates(start, end, months=tuple(range(1, 13))):
    """月末抽样日期。

    原为季频(3,6,9,12)，实测除息日距最近季度末权重中位数达50天，
    A股分红又集中在5-7月，季频权重系统性过期。改月频后中位数降到16天，
    成本约46万行（占2亿额度0.23%），值得。
    """
    dates = []
    for y in range(pd.Timestamp(start).year, pd.Timestamp(end).year + 1):
        for m in months:
            d = pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0)
            if pd.Timestamp(start) <= d <= pd.Timestamp(end):
                dates.append(d.strftime("%Y-%m-%d"))
    return dates


def fetch_weights(product, dates) -> pd.DataFrame:
    """get_index_weights 时点权重。中证系指数官方支持；
    上证50(000016)若取不到，回退用股本×价格自建市值权重"""
    idx = to_jq(PRODUCTS[product]["index"])
    frames = []
    for d in dates:
        try:
            w = jq.get_index_weights(idx, d)
        except Exception as e:
            print(f"    ⚠ [{product}] {d} 权重查询失败: {e}")
            continue
        if w is None or w.empty:
            continue
        frames.append(pd.DataFrame({
            "product": product, "date": d,
            "stock_code": [from_jq(c) for c in w.index],
            "weight": pd.to_numeric(w["weight"], errors="coerce"),
        }))
        time.sleep(0.1)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).dropna(subset=["weight"])


def fetch_financials(codes_jq, batch=50) -> pd.DataFrame:
    """归母净利润。用于德邦报告「路B」：分红总额 = 净利润 × 股利支付率

    关键是同时取 pub_date（财报披露日）。做point-in-time预测时必须按
    pub_date 过滤而不是 report_date —— 2023年报的report_date是2023-12-31，
    但要到2024年4月才公开，用report_date过滤会引入前视偏差。
    """
    frames, PAGE = [], 5000
    for i in range(0, len(codes_jq), batch):
        chunk = codes_jq[i:i + batch]
        off = 0
        while True:
            q = (query(finance.STK_INCOME_STATEMENT)
                 .filter(finance.STK_INCOME_STATEMENT.code.in_(chunk))
                 .order_by(finance.STK_INCOME_STATEMENT.id)
                 .limit(PAGE).offset(off))
            try:
                df = finance.run_query(q)
            except Exception as e:
                print(f"    ⚠ 财务批次{i//batch}失败: {e}")
                break
            if df is None or df.empty:
                break
            keep = [c for c in ["code", "report_date", "pub_date",
                                "np_parent_company_owners", "net_profit"] if c in df.columns]
            frames.append(df[keep])
            if len(df) < PAGE:
                break
            off += PAGE
            time.sleep(0.2)
        time.sleep(0.2)
    if not frames:
        return pd.DataFrame()
    d = pd.concat(frames, ignore_index=True)
    out = pd.DataFrame({
        "stock_code": d["code"].astype(str).map(from_jq),
        "report_date": pd.to_datetime(d["report_date"], errors="coerce").dt.strftime("%Y-%m-%d"),
        "pub_date": pd.to_datetime(d.get("pub_date"), errors="coerce").dt.strftime("%Y-%m-%d"),
        "np_parent": pd.to_numeric(d.get("np_parent_company_owners"), errors="coerce"),
        "net_profit": pd.to_numeric(d.get("net_profit"), errors="coerce"),
    }).dropna(subset=["report_date"])
    return out.drop_duplicates(["stock_code", "report_date"], keep="last")


def fetch_prices_grid(conn, product, codes, dates, batch_days=1) -> pd.DataFrame:
    """按「月末 + 除息日」这组日期，取成分股不复权收盘价

    只抓这两类日期而不是全日频：权重外推公式
        w_nt = w_n0 × (1+r_n) / Σ w_i0 × (1+r_i)
    只需要 基期(月末) 和 目标日(除息日) 两个时点的价格，中间日子用不到。
    约580万行 vs 全日频1075万行，精度一样，库体积减半。

    fq=None 不复权：算股息率 D/P 要的是除息日当天的原始价。
    库里已有的 (日期,股票) 跳过，可断点续跑。
    """
    if not codes or not dates:
        return pd.DataFrame()

    # 月末很多是周末/节假日（实测199个月末里69个非交易日），而月末正是权重基期。
    # 基期取不到价，r_n = P(除息日)/P(基期) 就算不出来 —— 统一回退到前一个交易日。
    from bisect import bisect_left
    cal = [r[0] for r in conn.execute("SELECT date FROM trading_calendar ORDER BY date")]
    cal_set = set(cal)
    aligned, moved = [], 0
    for d in dates:
        if d in cal_set:
            aligned.append(d)
        else:
            i = bisect_left(cal, d)
            if i:
                aligned.append(cal[i - 1]); moved += 1
    if moved:
        print(f"    [{product}] {moved} 个非交易日已回退到前一交易日")
    dates = sorted(set(aligned))

    # 跳过逻辑：按【该日期已有多少只股票】判断，不能只看"日期存在"。
    # 早期版本只抓除息日，每个日期仅1~2只股票；若仅凭日期存在就跳过，
    # 这些稀疏日期永远补不全，权重外推会因覆盖率不足而失效（实测500覆盖率0%）。
    # 门槛取 codes 的一半：早期未上市的股票本来就取不到，不能要求全有。
    cnt = dict(conn.execute(
        "SELECT date, COUNT(*) FROM constituent_daily WHERE product=? GROUP BY date",
        (product,)))
    need = max(1, int(len(codes) * 0.5))
    todo = [d for d in dates if cnt.get(d, 0) < need]
    thin = sum(1 for d in dates if 0 < cnt.get(d, 0) < need)
    if thin:
        print(f"    [{product}] {thin} 个日期数据过稀（<{need}只），将重新补全")
    if not todo:
        print(f"    [{product}] 价格已齐（{len(dates)}个日期），跳过")
        return pd.DataFrame()
    print(f"    [{product}] 需取 {len(todo)}/{len(dates)} 个日期 × {len(codes)} 只")

    frames, done = [], 0
    for d in todo:
        try:
            px = jq.get_price(codes, start_date=d, end_date=d, frequency="daily",
                              fields=["close"], fq=None, panel=False)
        except Exception as e:
            print(f"      ⚠ {d} 失败: {e}")
            continue
        if px is None or px.empty:
            continue
        px = px.reset_index()
        cc = "code" if "code" in px.columns else px.columns[1]
        frames.append(pd.DataFrame({
            "product": product, "date": d,
            "stock_code": px[cc].astype(str).map(from_jq),
            "close": pd.to_numeric(px["close"], errors="coerce"),
        }))
        done += 1
        if done % 100 == 0:
            # 分批落库，中途断了不用从头再来
            part = pd.concat(frames, ignore_index=True).dropna(subset=["close"])
            upsert(conn, "constituent_daily", part)
            print(f"      … 已完成 {done}/{len(todo)} 个日期，落库 {len(part)} 行")
            frames = []
        time.sleep(0.05)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).dropna(subset=["close"])

In [ ]:
def jq_probe(start="2023-01-01", end="2023-12-31"):
    """验证登录并打印接口字段，首次使用或聚宽改版时跑一下"""
    auth()
    print("\n【1】STK_XR_XD 分红表（贵州茅台样例）")
    try:
        q = query(finance.STK_XR_XD).filter(finance.STK_XR_XD.code == "600519.XSHG")
        try:
            df = finance.run_query(q)
        except Exception:
            df = jq.get_fundamentals(q)
        print(f"  全部列名: {list(df.columns)}")
        print(df.tail(3).to_string())
    except Exception as e:
        print(f"  ✗ 查询失败: {e}")

    print("\n【2】历史时点成分股 get_index_stocks")
    for p in ["300", "1000"]:
        try:
            codes = jq.get_index_stocks(to_jq(PRODUCTS[p]["index"]), "2023-06-30")
            print(f"  {p}: {len(codes)} 只，示例 {codes[:3]}")
        except Exception as e:
            print(f"  {p}: ✗ {e}")

    print("\n【3】时点权重 get_index_weights")
    for p in PRODUCTS:
        try:
            w = jq.get_index_weights(to_jq(PRODUCTS[p]["index"]), "2023-06-30")
            print(f"  {p}: {len(w) if w is not None else 0} 行"
                  + (f"，列 {list(w.columns)}" if w is not None and len(w) else ""))
        except Exception as e:
            print(f"  {p}: ✗ {e}")

    print("\n【4】估值表 valuation（总股本 capitalization）")
    try:
        v = jq.get_valuation("600519.XSHG", start, start, fields=["capitalization", "circulating_cap"])
        print(v.to_string())
    except Exception as e:
        print(f"  ✗ {e}")


def jq_backfill(conn, products=None, start=None, end=None, full=False, dry_run=False):
    """增量（或全量）回填：成分股 → 分红事件/在途预案 → 月频权重 → 价格网格 → 财务"""
    auth()
    today = datetime.now().strftime("%Y-%m-%d")
    end = end or today
    user_start = start

    for p in parse_products(products):
        cfg = PRODUCTS[p]
        if user_start or full:
            start = user_start or "2010-01-01"
        else:
            # 增量：从水位往回退一段重取。不能直接用 MAX(ex_date) 的次日做起点 ——
            # 库里存着已公告的【未来】除息日，水位会跑到今天之后，导致
            # ① 水位之前才补全实施公告日的事件永远不会更新；② 新公告的、除息日早于水位的事件被漏掉
            wm = min(last_date(conn, "dividend_announcement", p) or "2010-01-01", end)
            start = max("2010-01-01", (pd.Timestamp(wm) - pd.Timedelta(days=INCREMENTAL_LOOKBACK_DAYS))
                        .strftime("%Y-%m-%d"))
        print(f"\n[{cfg['label']} {p}] {start} ~ {end}")
        if start > end:
            print("  已最新，跳过"); continue

        dates = sample_dates(start, end)
        if dry_run:
            print(f"  dry-run: 成分股/权重抽样 {len(dates)} 个月末"); continue

        # 1) 成分股名单（圈定分红事件范围）
        unions = {}
        for d in dates:
            try:
                unions[d] = [from_jq(c) for c in jq.get_index_stocks(to_jq(cfg["index"]), d)]
            except Exception as e:
                print(f"  ⚠ {d} 成分股失败: {e}")
            time.sleep(0.1)
        all_codes = sorted({c for v in unions.values() for c in v})
        print(f"  成分股抽样 {len(unions)} 期，去重后 {len(all_codes)} 只")

        # 2) 分红事件。上界放宽一年：已发实施公告、除息日在未来的事件对当下预测最有用
        ev_end = (pd.Timestamp(end) + pd.Timedelta(days=366)).strftime("%Y-%m-%d")
        pending_since = (pd.Timestamp(today) - pd.Timedelta(days=730)).strftime("%Y-%m-%d")
        ev, pend = fetch_dividends([to_jq(c) for c in all_codes], start, ev_end, pending_since)
        # 在途预案是「当前状态」快照：按品种整体替换，已定除息日的自然就移出去了
        conn.execute("DELETE FROM dividend_pending WHERE product=?", (p,))
        if len(pend):
            pend = pend.assign(product=p, snapshot_date=today)
            n = upsert(conn, "dividend_pending", pend)
            print(f"  在途预案 → dividend_pending 写入 {n} 行；进度分布: "
                  f"{pend['stage'].value_counts().head(6).to_dict()}")
        conn.commit()
        if ev.empty:
            print("  ⚠ 无分红事件"); continue
        ev["product"] = p
        ev["snapshot_date"] = today
        cols = [c for c in ["product", "stock_code", "ex_date", "div_per_share",
                            "div_total", "shares", "stage", "plan_date", "sgm_date",
                            "impl_date", "reg_date", "report_date", "snapshot_date"]
                if c in ev.columns]
        n = upsert(conn, "dividend_announcement", ev[cols])
        print(f"  分红事件 → dividend_announcement 写入 {n} 行")
        for c in ["sgm_date", "impl_date", "div_total"]:
            if c in ev.columns:
                print(f"      {c} 非空率 {ev[c].notna().mean()*100:.1f}%")

        # 3) 月频权重
        w = fetch_weights(p, list(unions.keys()))
        if len(w):
            n = upsert(conn, "index_weight", w)
            print(f"  月频权重 → index_weight 写入 {n} 行")
        elif unions:
            print("  ⚠ 权重取空")

        # 4) 价格网格：月末(权重基期) + 已发生的除息日(目标日)
        exd = [d for d in ev["ex_date"].dropna() if d <= end]
        grid = sorted(set(dates) | set(exd))
        px = fetch_prices_grid(conn, p, [to_jq(c) for c in all_codes], grid)
        if len(px):
            n = upsert(conn, "constituent_daily", px)
            print(f"  价格网格 → constituent_daily 写入 {n} 行")

    # 5) 财务数据：跨品种共享，按全部出现过的成分股一次取完
    if not dry_run:
        codes = pd.read_sql(
            "SELECT DISTINCT stock_code FROM dividend_announcement", conn)["stock_code"].tolist()
        if codes:
            print(f"\n[归母净利润] {len(codes)} 只股票")
            fin = fetch_financials([to_jq(c) for c in codes])
            if len(fin):
                n = upsert(conn, "stock_financials", fin)
                print(f"  → stock_financials 写入 {n} 行")
    return status(conn)

In [ ]:
if RUN_JQ:
    load_env()
    jq_backfill(conn, full=JQ_FULL)

## 三、数据体检

- 公告日字段非空率（sgm/impl 不会是 100%，但接近 0% 说明字段映射有问题）
- 权重是否月频
- 股本单位（以贵州茅台为准，总股本应在 12.5 亿股量级）

In [ ]:
def check(conn, p):
    print("=" * 66)
    print(f"【1】公告日字段非空率  product={p}")
    d = pd.read_sql("SELECT * FROM dividend_announcement WHERE product=?", conn, params=[p])
    print(f"  总行数 {len(d):,}")
    for col in ["plan_date", "sgm_date", "impl_date", "reg_date", "report_date",
                "div_per_share", "shares", "div_total"]:
        if col in d.columns:
            nn = d[col].notna().sum()
            print(f"    {col:<15} {nn:>7,}  ({nn/max(len(d),1)*100:5.1f}%)")
    print("  判读：sgm_date/impl_date 不会是100%（进行中的分红还没走完流程），")
    print("        但若接近0%，说明字段映射有问题。")

    print("\n" + "=" * 66)
    print(f"【2】权重频率  product={p}")
    w = pd.read_sql(
        "SELECT date, COUNT(*) n, ROUND(SUM(weight),3) sum_w "
        "FROM index_weight WHERE product=? GROUP BY date ORDER BY date", conn, params=[p])
    print(f"  共 {len(w)} 个时点")
    recent = w.tail(14)
    print(recent.to_string(index=False))
    months = pd.to_datetime(w["date"]).dt.month.unique()
    print(f"\n  出现过的月份: {sorted(months)}")
    print(f"  → {'✓ 月频生效' if len(set(months)) > 4 else '✗ 仍是季频，需要用价格网格做权重外推'}")
    if len(recent) >= 2:
        same = recent["sum_w"].nunique() == 1 and recent["n"].nunique() == 1
        print(f"  相邻时点权重合计是否完全相同: {'是（可疑，可能是同一份快照）' if same else '否（正常）'}")

    print("\n" + "=" * 66)
    print("【3】股本单位（以贵州茅台为准，总股本应在 12.5 亿股量级）")
    t = pd.read_sql(
        "SELECT stock_code, ex_date, div_per_share, shares, div_total "
        "FROM dividend_announcement WHERE stock_code LIKE '600519%' "
        "ORDER BY ex_date DESC LIMIT 5", conn)
    if t.empty:
        print("  没找到茅台记录（可能该品种不含它）")
    else:
        print(t.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
        s = t["shares"].dropna()
        if len(s):
            v = s.iloc[0]
            if 5e8 < v < 5e9:
                print(f"\n  ✓ shares={v:,.0f} 在 12.5亿股 量级，单位正确")
            elif 5e4 < v < 5e5:
                print(f"\n  ✗ shares={v:,.0f} 看着是【万股】，×1e4 那步没生效")
            else:
                print(f"\n  ⚠ shares={v:,.0f} 量级异常，需人工确认")
        dt = t["div_total"].dropna()
        if len(dt):
            print(f"  div_total 首行 {dt.iloc[0]:,.0f} 元 —— 茅台应在【几百亿】量级")

    print("\n" + "=" * 66)
    print("【附】价格网格进度")
    g = pd.read_sql(
        "SELECT COUNT(*) rows, COUNT(DISTINCT date) dates, MIN(date) a, MAX(date) b "
        "FROM constituent_daily WHERE product=?", conn, params=[p])
    print(g.to_string(index=False))

In [ ]:
check(conn, "300")

In [ ]:
# 删除老库里的弃用表和冗余索引并压缩（需要约等于库大小的临时磁盘空间）
# conn.close(); vacuum(DB_PATH); conn = connect(DB_PATH)